In [ ]:
# Notebook cell: batch conversions between (ε,δ)-DP, μ-GDP, and (α, ε_α)-RDP

import math
from typing import Iterable, Optional

# =========================
# Normal CDF
# =========================
def normal_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# =========================
# GDP formulas
# =========================
def delta_from_eps_mu(eps: float, mu: float) -> float:
    """
    δ(ε; μ) = Φ(-ε/μ + μ/2) - exp(ε) Φ(-ε/μ - μ/2)
    """
    if mu <= 0:
        raise ValueError("mu must be > 0")
    term1 = normal_cdf(-eps / mu + mu / 2.0)
    term2 = math.exp(eps) * normal_cdf(-eps / mu - mu / 2.0)
    delta = term1 - term2
    return max(0.0, min(1.0, delta))

def mu_from_eps_delta(eps: float, delta_target: float, tol: float = 1e-12, max_iter: int = 300) -> float:
    """
    Solve for μ given ε and δ using bisection.
    """
    if eps < 0:
        raise ValueError("eps must be >= 0")
    if not (0 < delta_target < 1):
        raise ValueError("delta_target must be in (0,1)")

    lo = 1e-12
    hi = 1.0

    while delta_from_eps_mu(eps, hi) < delta_target:
        hi *= 2.0
        if hi > 1e6:
            raise RuntimeError(f"Could not bracket mu for eps={eps}, delta={delta_target}")

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        val = delta_from_eps_mu(eps, mid)
        if abs(val - delta_target) < tol:
            return mid
        if val < delta_target:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)

def eps_from_mu_delta(mu: float, delta_target: float, tol: float = 1e-12, max_iter: int = 300) -> float:
    """
    Solve for ε given μ and δ using bisection.
    """
    if mu <= 0:
        raise ValueError("mu must be > 0")
    if not (0 < delta_target < 1):
        raise ValueError("delta_target must be in (0,1)")

    lo = 0.0
    hi = 1.0

    while delta_from_eps_mu(hi, mu) > delta_target:
        hi *= 2.0
        if hi > 1e6:
            raise RuntimeError(f"Could not bracket epsilon for mu={mu}, delta={delta_target}")

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        val = delta_from_eps_mu(mid, mu)
        if abs(val - delta_target) < tol:
            return mid
        if val > delta_target:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)

def rdp_eps_from_mu_alpha(mu: float, alpha: float) -> float:
    """
    μ-GDP => (α, ε_α)-RDP with ε_α = α μ^2 / 2
    """
    if mu <= 0:
        raise ValueError("mu must be > 0")
    if alpha <= 1:
        raise ValueError("alpha must be > 1")
    return 0.5 * alpha * mu * mu

# =========================
# Pretty printer
# =========================
def print_table(rows, headers, float_digits=10):
    widths = []
    for i, h in enumerate(headers):
        col_vals = [h]
        for row in rows:
            v = row[i]
            if isinstance(v, float):
                col_vals.append(f"{v:.{float_digits}g}")
            else:
                col_vals.append(str(v))
        widths.append(max(len(x) for x in col_vals))

    def fmt(v, w):
        if isinstance(v, float):
            s = f"{v:.{float_digits}g}"
        else:
            s = str(v)
        return s.ljust(w)

    print(" | ".join(fmt(h, w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(fmt(v, w) for v, w in zip(row, widths)))

# =========================
# Batch converters
# =========================
def convert_eps_list_to_mu_rdp(
    eps_list: Iterable[float],
    delta: float,
    alpha: float = 2.0
):
    rows = []
    for eps in eps_list:
        mu = mu_from_eps_delta(eps, delta)
        rdp = rdp_eps_from_mu_alpha(mu, alpha)
        rows.append((eps, delta, mu, alpha, rdp))
    headers = ["eps", "delta", "mu", "alpha", "RDP_eps_alpha"]
    print("\nFrom (eps, delta) to mu-GDP and RDP:\n")
    print_table(rows, headers)
    return rows

def convert_mu_list_to_eps_rdp(
    mu_list: Iterable[float],
    delta: float,
    alpha: float = 2.0
):
    rows = []
    for mu in mu_list:
        eps = eps_from_mu_delta(mu, delta)
        rdp = rdp_eps_from_mu_alpha(mu, alpha)
        rows.append((mu, delta, eps, alpha, rdp))
    headers = ["mu", "delta", "eps_at_delta", "alpha", "RDP_eps_alpha"]
    print("\nFrom mu-GDP to (eps, delta)-DP and RDP:\n")
    print_table(rows, headers)
    return rows

# =========================
# EDIT THESE INPUTS
# =========================

# Case 1: insert eps values here
eps_values = [1.5, 2.2, 2.6]
delta = 1e-5
alpha = 2.0

# Case 2: insert mu values here
mu_values = [0.5, 1.0, math.sqrt(2), 2.0, math.sqrt(10)]

# =========================
# RUN BOTH
# =========================
eps_rows = convert_eps_list_to_mu_rdp(eps_values, delta=delta, alpha=alpha)
mu_rows  = convert_mu_list_to_eps_rdp(mu_values,  delta=delta, alpha=alpha)


From (eps, delta) to mu-GDP and RDP:

eps   | delta | mu           | alpha | RDP_eps_alpha
------+-------+--------------+-------+--------------
2     | 1e-05 | 0.5015516877 | 3     | 0.3773311432 
4.38  | 1e-05 | 1.000556797  | 3     | 1.501670855  
6.57  | 1e-05 | 1.413676493  | 3     | 2.99772184   
10    | 1e-05 | 2.000445619  | 3     | 6.002674013  
17.85 | 1e-05 | 3.161382064  | 3     | 14.99150483  

From mu-GDP to (eps, delta)-DP and RDP:

mu          | delta | eps_at_delta | alpha | RDP_eps_alpha
------------+-------+--------------+-------+--------------
0.5         | 1e-05 | 1.993091404  | 3     | 0.375        
1           | 1e-05 | 4.377178073  | 3     | 1.5          
1.414213562 | 1e-05 | 6.572970092  | 3     | 3            
2           | 1e-05 | 9.99725616   | 3     | 6            
3.16227766  | 1e-05 | 17.85656089  | 3     | 15           


In [2]:

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_gpus = 1 if device=='cuda' else 0
print(device)

cpu


In [3]:
import numpy as np
from pathlib import Path

# Choose the parent folder and run name.
# Expected layout for this experiment:
# exp_data/max_grad_norm/1.0/seed5/cifar10_half_cnn_eps2.0/losses_in.npy
# exp_data/max_grad_norm/1.0/seed6/cifar10_half_cnn_eps2.0/losses_out.npy
results_root = Path("exp_data/max_grad_norm/1.0")
run_name = "cifar10_half_cnn_eps2.0"
seeds = [5, 6, 7, 8, 9]

all_losses_in = []
all_losses_out = []
emp_losses = []

for seed in seeds:
    base = results_root / f"seed{seed}" / run_name
    in_path = base / "losses_in.npy"
    out_path = base / "losses_out.npy"

    if not in_path.exists() or not out_path.exists():
        raise FileNotFoundError(f"Missing losses for seed {seed}: {base}")

    seed_losses_in = np.load(in_path).reshape(-1)
    seed_losses_out = np.load(out_path).reshape(-1)

    print(f"seed {seed}: losses_in {seed_losses_in.shape}, losses_out {seed_losses_out.shape}")

    all_losses_in.append(seed_losses_in)
    all_losses_out.append(seed_losses_out)
    emp_losses.append(np.mean(np.concatenate([seed_losses_in, seed_losses_out])))

losses_in = np.concatenate(all_losses_in)
losses_out = np.concatenate(all_losses_out)

print("\nConcatenated across seeds 5-9")
print("losses_in:", losses_in.shape, losses_in.dtype)
print("losses_out:", losses_out.shape, losses_out.dtype)
print("mean empirical loss across the five seeds:", float(np.mean(emp_losses)))
print(losses_in)
print(losses_out)

seed 5: losses_in (100,), losses_out (100,)
seed 6: losses_in (100,), losses_out (100,)
seed 7: losses_in (100,), losses_out (100,)
seed 8: losses_in (100,), losses_out (100,)
seed 9: losses_in (100,), losses_out (100,)

Concatenated across seeds 5-9
losses_in: (500,) float64
losses_out: (500,) float64
mean empirical loss across the five seeds: -5.793532562851905
[-6.58754015 -8.18331814 -6.59491396 -7.10351562 -7.03113317 -5.69584846
 -6.15520239 -5.49449348 -5.77564478 -7.62563276 -5.11787748 -7.17541504
 -4.3843441  -6.15557003 -6.93572426 -7.87129116 -7.7116766  -5.29892063
 -5.45697212 -4.63691902 -7.34587812 -3.22857189 -7.58277607 -6.88469744
 -5.24616289 -6.15069246 -3.68253827 -5.59923697 -7.39948273 -7.77665281
 -5.21402884 -6.48097181 -6.47802305 -4.79314041 -4.53118181 -8.46925545
 -7.56920815 -6.86877012 -7.81023026 -6.59770203 -6.72559643 -8.46512127
 -6.94296932 -7.73461771 -6.54601288 -4.54303741 -6.32967901 -4.68229818
 -5.84985495 -6.33932924 -6.61271143 -6.84363699 -

In [4]:
class LoadDataset(torch.utils.data.Dataset):
    def __init__(self, N, x, y, x_dim, z_dim):
        self.x = x
        self.y = y
        # file imports and object initialization
        
        self.N = N
        self.x_dim = x_dim
        self.z_dim = z_dim

    def __getitem__(self, ix):
        a, b = self.x[ix,0:self.x_dim], self.y[ix,0:self.z_dim]
        
        return a, b
    
    def __len__(self):
        return self.N

In [5]:
%reload_ext autoreload
from renyi import RenyiDivergenceEstimator
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import TQDMProgressBar

x_dim = 1 #plaintext length of a single sample
z_dim = 1 #ciphertext length of a single sample
N = 49999 # num samples -1
lr = 1e-4 #learning rate
epochs = 10 #epochs
batch_size = 5000 #batch size
Renyi_order = 3
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
devices = 1  # one device either way

from pytorch_lightning.loggers import CSVLogger

csv_logger = CSVLogger("lightning_logs", name="renyi_run")

#format data
loss_type = ['mine']
# X, Y already constructed as tensors of shape (N, 1)
X = torch.tensor(losses_in,  dtype=torch.float32).view(-1, 1)
Y = torch.tensor(losses_out, dtype=torch.float32).view(-1, 1)
N = min(len(X), len(Y))
X, Y = X[:N], Y[:N]  # ensure same length

# --- 80/20 split: first 80% train, last 20% test ---
n_train = int(0.8 * N)
X_train, Y_train = X[:n_train], Y[:n_train]
X_test,  Y_test  = X[n_train:], Y[n_train:]

train_loader = torch.utils.data.DataLoader(
    LoadDataset(len(X_train), X_train, Y_train, x_dim=1, z_dim=1),
    batch_size=batch_size,
    shuffle=True,      # ok to shuffle *within* the training set
    drop_last=False
)

test_loader = torch.utils.data.DataLoader(
    LoadDataset(len(X_test), X_test, Y_test, x_dim=1, z_dim=1),
    batch_size=batch_size,
    shuffle=False,
    drop_last=False
)



trainer = Trainer(
    max_epochs=epochs,
    accelerator=accelerator,
    devices=devices,
    logger=csv_logger,
    enable_checkpointing=False,
    enable_progress_bar=True,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    log_every_n_steps=10,
)
model = RenyiDivergenceEstimator(
    lr=lr,
    renyi_order=Renyi_order,
    ema_rate=0.25,   # your EMA rate (beta)
    hidden=100,
)


trainer.fit(model, train_dataloaders=train_loader)
results = trainer.test(model, dataloaders=test_loader)

print("DV-Rényi lower bound (epoch-avg):", results[0]["test_renyi_lb"])
print("Same value from model:", model.avg_test_renyi_lb)

/u/bdkim4/.conda/envs/bb_audit_dpsgd/lib/python3.10/site-packages/sympy/external/gmpy.py:138: UserWarning: gmpy2 version is too old to use (2.0.0 or newer required)
  gmpy = import_module('gmpy2', min_module_version=_GMPY2_MIN_VERSION,


ModuleNotFoundError: No module named 'pkg_resources'

In [6]:
import sys
print(sys.executable)

import torch
print(torch.__version__)
print(torch.cuda.is_available())


/u/bdkim4/.conda/envs/bb_audit_dpsgd/bin/python
2.5.1+cu121
False


/u/bdkim4/.conda/envs/bb_audit_dpsgd/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [7]:
import pytorch_lightning as pl
print(pl.__version__)


ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
import numpy as np
import math

Renyi_order = 2.0
# -------------------------
# Utilities
# -------------------------
def _to_1d_finite(x):
    x = np.asarray(x).reshape(-1).astype(np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        raise ValueError("No finite samples after filtering NaN/Inf.")
    return x

def logmeanexp(a):
    a = np.asarray(a, dtype=np.float64)
    m = np.max(a)
    return m + np.log(np.mean(np.exp(a - m)))

# -------------------------
# Gaussian MLE fit + logpdf
# -------------------------
def fit_gaussian_mle(x, var_floor=1e-12):
    x = _to_1d_finite(x)
    mu = float(np.mean(x))
    var = float(np.mean((x - mu) ** 2))  # MLE (divide by n)
    var = max(var, var_floor)
    return mu, var

def logpdf_gaussian(x, mu, var):
    x = np.asarray(x, dtype=np.float64)
    return -0.5 * (np.log(2.0 * np.pi * var) + (x - mu) ** 2 / var)

# -------------------------
# Rényi divergence estimators
#   D_alpha(Q||P) = 1/(alpha-1) * log ∫ q(x)^alpha p(x)^(1-alpha) dx
# -------------------------
def renyi_gaussian_closed_form(mu_q, var_q, mu_p, var_p, alpha):
    """
    Closed-form for 1D Gaussian q=N(mu_q,var_q), p=N(mu_p,var_p).
    Returns +inf if the integral diverges (can happen for alpha>1 if p is too "narrow").
    """
    a = float(alpha)
    if a <= 1.0:
        raise ValueError("alpha must be > 1")

    # Derived by completing the square in the exponent:
    # A = a/var_q - (a-1)/var_p must be > 0 for finiteness.
    A = a / var_q - (a - 1.0) / var_p
    if A <= 0:
        return float("inf")

    B = a * mu_q / var_q - (a - 1.0) * mu_p / var_p
    C = a * (mu_q ** 2) / var_q - (a - 1.0) * (mu_p ** 2) / var_p

    # log I where I = ∫ q^a p^(1-a)
    logI = (
        -0.5 * a * np.log(var_q)
        -0.5 * (1.0 - a) * np.log(var_p)
        -0.5 * np.log(A)
        -0.5 * (C - (B * B) / A)
    )
    return float(logI / (a - 1.0))

def renyi_gaussian_mle_closed(q_samples, p_samples, alpha):
    mu_q, var_q = fit_gaussian_mle(q_samples)
    mu_p, var_p = fit_gaussian_mle(p_samples)
    D = renyi_gaussian_closed_form(mu_q, var_q, mu_p, var_p, alpha)
    return D, (mu_q, var_q, mu_p, var_p)

def renyi_gaussian_mle_mc(q_samples, p_samples, alpha):
    """
    MLE-fit Gaussians to Q and P, then estimate:
      D_alpha(Q||P) = (1/(a-1)) log E_{X~P}[ (q(X)/p(X))^a ]
    using X as your *empirical* p_samples (MC under P).
    """
    a = float(alpha)
    if a <= 1.0:
        raise ValueError("alpha must be > 1")

    q = _to_1d_finite(q_samples)
    p = _to_1d_finite(p_samples)

    mu_q, var_q = fit_gaussian_mle(q)
    mu_p, var_p = fit_gaussian_mle(p)

    log_q = logpdf_gaussian(p, mu_q, var_q)
    log_p = logpdf_gaussian(p, mu_p, var_p)

    # log w_i = a*(log q(x_i) - log p(x_i))
    logw = a * (log_q - log_p)
    logE = logmeanexp(logw)  # log E_P[(q/p)^a]
    return float(logE / (a - 1.0)), (mu_q, var_q, mu_p, var_p)

# -------------------------
# Optional: GMM MLE + MC (if sklearn is available)
# -------------------------
def renyi_gmm_mle_mc(q_samples, p_samples, alpha, n_components=3, reg_covar=1e-6, seed=0):
    """
    Fits separate 1D GMMs to Q and P by EM (sklearn), then uses empirical P samples for MC:
      D_alpha(Q||P) = (1/(a-1)) log E_{X~P}[ (q_hat(X)/p_hat(X))^a ].
    """
    a = float(alpha)
    if a <= 1.0:
        raise ValueError("alpha must be > 1")

    q = _to_1d_finite(q_samples).reshape(-1, 1)
    p = _to_1d_finite(p_samples).reshape(-1, 1)

    try:
        from sklearn.mixture import GaussianMixture
    except Exception as e:
        raise ImportError("scikit-learn not available; install or skip GMM.") from e

    gq = GaussianMixture(n_components=n_components, reg_covar=reg_covar, random_state=seed)
    gp = GaussianMixture(n_components=n_components, reg_covar=reg_covar, random_state=seed)
    gq.fit(q)
    gp.fit(p)

    # score_samples gives log density
    log_q = gq.score_samples(p)
    log_p = gp.score_samples(p)

    logw = a * (log_q - log_p)
    logE = logmeanexp(logw)
    return float(logE / (a - 1.0))

# -------------------------
# Optional: bootstrap CI (rough)
# -------------------------
def bootstrap_ci(estimator_fn, q_samples, p_samples, alpha, B=200, seed=0):
    q = _to_1d_finite(q_samples)
    p = _to_1d_finite(p_samples)
    rng = np.random.default_rng(seed)

    n_q, n_p = len(q), len(p)
    ests = np.empty(B, dtype=np.float64)
    for b in range(B):
        qb = q[rng.integers(0, n_q, size=n_q)]
        pb = p[rng.integers(0, n_p, size=n_p)]
        ests[b] = estimator_fn(qb, pb, alpha)[0] if isinstance(estimator_fn(qb, pb, alpha), tuple) else estimator_fn(qb, pb, alpha)

    est = float(np.mean(ests))
    lo, hi = np.quantile(ests, [0.025, 0.975])
    return est, (float(lo), float(hi))

# -------------------------
# Run on your arrays
# -------------------------
alpha = float(Renyi_order)  # e.g., 2.0

Q = losses_in   # treat as Q samples
P = losses_out  # treat as P samples

D_gauss_closed, params = renyi_gaussian_mle_closed(Q, P, alpha)
D_gauss_mc,     params2 = renyi_gaussian_mle_mc(Q, P, alpha)

print("\n=== Gaussian MLE results ===")
print("params (mu_q, var_q, mu_p, var_p):", params)
print(f"D_{alpha}(Q||P) Gaussian closed-form  : {D_gauss_closed:.6f}")
print(f"D_{alpha}(Q||P) Gaussian MLE + MC      : {D_gauss_mc:.6f}")

# Optional: GMM
try:
    D_gmm_mc = renyi_gmm_mle_mc(Q, P, alpha, n_components=3)
    print("\n=== GMM(3) MLE + MC ===")
    print(f"D_{alpha}(Q||P) GMM(3) MLE + MC        : {D_gmm_mc:.6f}")
except ImportError as e:
    print("\n(skipping GMM; scikit-learn not available)")

# Optional: bootstrap CI (on Gaussian closed-form plug-in)
est_mean, (ci_lo, ci_hi) = bootstrap_ci(renyi_gaussian_mle_closed, Q, P, alpha, B=200, seed=0)
print("\n=== Rough bootstrap (Gaussian plug-in) ===")
print(f"bootstrap mean: {est_mean:.6f}  95% CI: [{ci_lo:.6f}, {ci_hi:.6f}]")



alpha = float(Renyi_order)  # e.g., 2.0

Q = losses_out   # treat as Q samples
P = losses_in  # treat as P samples

D_gauss_closed, params = renyi_gaussian_mle_closed(Q, P, alpha)
D_gauss_mc,     params2 = renyi_gaussian_mle_mc(Q, P, alpha)

print("\n=== Gaussian MLE results ===")
print("params (mu_q, var_q, mu_p, var_p):", params)
print(f"D_{alpha}(P||Q) Gaussian closed-form  : {D_gauss_closed:.6f}")
print(f"D_{alpha}(P||Q) Gaussian MLE + MC      : {D_gauss_mc:.6f}")

# Optional: GMM
try:
    D_gmm_mc = renyi_gmm_mle_mc(Q, P, alpha, n_components=3)
    print("\n=== GMM(3) MLE + MC ===")
    print(f"D_{alpha}(P||Q) GMM(3) MLE + MC        : {D_gmm_mc:.6f}")
except ImportError as e:
    print("\n(skipping GMM; scikit-learn not available)")

# Optional: bootstrap CI (on Gaussian closed-form plug-in)
est_mean, (ci_lo, ci_hi) = bootstrap_ci(renyi_gaussian_mle_closed, Q, P, alpha, B=200, seed=0)
print("\n=== Rough bootstrap (Gaussian plug-in) ===")
print(f"bootstrap mean: {est_mean:.6f}  95% CI: [{ci_lo:.6f}, {ci_hi:.6f}]")